# Task 2 — Classificatore manuale: **1R (1-Rule)**

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)
**Dataset:** `manuale.csv` (12 campioni, bilanciati 6 `yes` / 6 `no`)


> 📄 Documentazione discorsiva e motivazioni di progetto: [`docs/task2.md`](../docs/task2.md)

In questo notebook costruiamo **a mano**, passo dopo passo, il classificatore **1R** sul file
`manuale.csv`. Come richiesto, ne illustriamo l'adattamento ai
dati, lo implementiamo in Python e ne valutiamo le prestazioni **sullo stesso file
`manuale.csv`**.

1R è il classificatore più semplice possibile e funge da **baseline** da confrontare con il
Naïve Bayes (notebook `task2_NaiveBayes.ipynb`).

## 1. Caricamento del dataset `manuale.csv`

Carichiamo il file `manuale.csv` estratto nel Task 1: **12 campioni bilanciati** rispetto alla
target `y` (`1` = deposito sottoscritto, `0` = non sottoscritto).

In [1]:
import pandas as pd

dataFrame_manuale = pd.read_csv("../data/processed/manuale.csv" , sep=";")
dataFrame_manuale

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## 2. Separazione tra feature e target, tipi di attributo

Distinguiamo gli attributi **nominali** da quelli **numerici**: 1R li tratta diversamente,
perché i numerici richiedono una **discretizzazione** preliminare.

In [2]:
valori_target = dataFrame_manuale["y"]

nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]

print("Attributi nominali:", nominali)
print("Attributi numerici:", numerici)
print("\nDistribuzione della classe target:")
print(valori_target.value_counts())

Attributi nominali: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
Attributi numerici: ['age', 'campaign']

Distribuzione della classe target:
y
1    6
0    6
Name: count, dtype: int64


## 3. Richiamo teorico: come funziona 1R

**1R** (*One Rule*) costruisce un **albero decisionale a un solo livello**: classifica usando
**un unico attributo**, quello che da solo produce il minor numero di errori. La procedura è:

1. **per ogni attributo** si costruisce una regola: per ciascun valore dell'attributo si
   guarda **quale classe è più frequente** tra le istanze con quel valore, e si assegna quella
   classe a quel valore;
2. si **contano gli errori** che la regola commette sul training set (le istanze che
   ricadono nella classe di minoranza per il loro valore);
3. si **sceglie l'attributo** la cui regola produce il **minor numero di errori totali**.

Per gli **attributi numerici** 1R richiede prima una **discretizzazione** in intervalli (*bin*).
Qui usiamo un binning semplice sulla **mediana**: due intervalli, `<= mediana` e `> mediana`.

> **Punto critico (dalle slide).** 1R tende all'**overfitting** quando un attributo ha **molti
> valori distinti**: al limite, un attributo con un valore diverso per ogni istanza darebbe
> **0 errori** sul training, ma sarebbe del tutto inutile in generalizzazione. Lo vedremo
> concretamente sui nostri dati.

## 4. Adattamento ai dati: calcolo degli errori per ogni attributo

Definiamo una funzione che, dato un attributo, costruisce la regola 1R e ne conta gli errori.
Usiamo `crosstab` per la tabella valore × classe: per ogni valore, la **classe assegnata** è
quella di maggioranza (`idxmax`) e gli **errori** sono `(totale del valore − conteggio della
classe di maggioranza)`, sommati su tutti i valori.

In [3]:

#implementa 1R per un attributo nominale, restituendo le regole e il numero di errori
def regole_e_errori_attributo(attributo, y):
    tabella = pd.crosstab(attributo, y)            # tabella che per ogni valore dell'attributo conta le occorrenze di ciascuna classe target
    regole = tabella.idxmax(axis=1).to_dict()       # per ogni valore dell attributo restituisce la classe target con piu occorrenze (dizionario che ha come chiave il valore e come attributo la classe target)
    errori = int((tabella.sum(axis=1) - tabella.max(axis=1)).sum()) # per ogni valore dell'attributo calcola il numero di errori (totale - max) e somma tutti gli errori
    return regole, errori

### 4.1 Errori sugli attributi nominali

Applichiamo la funzione a tutti i nominali tramite `apply` su una `Series` (niente `for`).

In [4]:
#applica 1R a tutti gli attributi nominali 
def errori_nominale(attributo):
    regole, errori = regole_e_errori_attributo(dataFrame_manuale[attributo], valori_target)
    print(f"{attributo:12s}: {errori}/12 errori   regole={regole}")
    return errori
    
errori_e_regole_attributiNominali = pd.Series(nominali, index=nominali).apply(errori_nominale) #stampa regole e errori per tutti gli attributi nominali


job         : 2/12 errori   regole={'admin.': 1, 'blue-collar': 0, 'retired': 1, 'student': 1, 'technician': 1, 'unknown': 0}
marital     : 2/12 errori   regole={'divorced': 1, 'married': 0, 'single': 1}
education   : 3/12 errori   regole={'basic.4y': 1, 'basic.6y': 0, 'basic.9y': 1, 'high.school': 0, 'professional.course': 1, 'university.degree': 0, 'unknown': 0}
housing     : 5/12 errori   regole={'no': 0, 'unknown': 1, 'yes': 0}
loan        : 5/12 errori   regole={'no': 0, 'unknown': 1, 'yes': 0}
contact     : 5/12 errori   regole={'cellular': 1, 'telephone': 0}
poutcome    : 6/12 errori   regole={'failure': 0, 'nonexistent': 0}


### 4.2 Errori sugli attributi numerici (discretizzati per mediana)

I numerici `age` e `campaign` vengono prima discretizzati in due fasce sulla **mediana**, poi
trattati come i nominali.

In [5]:
#Gestione degli attributi numerici: per ogni attributo numerico calcola la mediana e divide i valori in due gruppi (<= mediana e > mediana). Poi applica 1R a questi due gruppi e stampa le regole e il numero di errori.
def errori_numerico(attributo):
    med = dataFrame_manuale[attributo].median()
    binned = (dataFrame_manuale[attributo] > med).map({False: f"<= {med}", True: f"> {med}"})
    regole, errori = regole_e_errori_attributo(binned, valori_target)
    print(f"{attributo:12s} (mediana={med}): {errori}/12 errori   regole={regole}")
    return errori

errori_e_regole_attributiNumerici = pd.Series(numerici, index=numerici).apply(errori_numerico)

age          (mediana=36.0): 5/12 errori   regole={'<= 36.0': 1, '> 36.0': 0}
campaign     (mediana=2.5): 4/12 errori   regole={'<= 2.5': 1, '> 2.5': 0}


### 4.3 Scelta dell'attributo migliore

Mettiamo insieme gli errori di tutti gli attributi e scegliamo quello con il **minimo**.

In [6]:
errori_e_regole_tutti = pd.concat([errori_e_regole_attributiNominali, errori_e_regole_attributiNumerici]).sort_values()
migliore = errori_e_regole_tutti.idxmin()

print(errori_e_regole_tutti.to_string())
print(f"\n=> 1R sceglie '{migliore}' con {errori_e_regole_tutti[migliore]}/12 errori")



job          2
marital      2
education    3
campaign     4
loan         5
contact      5
age          5
housing      5
poutcome     6

=> 1R sceglie 'job' con 2/12 errori


### Osservazione: un pareggio rivelatore

`job` e `marital` producono **entrambi 2 errori**: 1R sceglie il primo per semplice ordine
(*tie-breaking*). Ma c'è una differenza sostanziale:

- `job` ha **6 valori**, di cui **4 con una sola istanza** (`retired`, `student`, `technician`,
  `unknown`)
- `marital` ha solo **3 valori** e raggiunge gli stessi 2 errori in modo molto più **robusto**.
Scegliento `job` abbiamo aumentato **l'overfitting**.

Possiamo concludere che parità di errori sul training, `marital` sarebbe la scelta più affidabile.

## 5. Implementazione del classificatore 1R

Costruiamo le due funzioni del classificatore:

- `addestra_1R`: dato il training set, calcola gli errori di tutti gli attributi, sceglie il
  migliore e ne memorizza la regola (gestendo eventualmente la soglia per i numerici);
- `predici_1R`: applica la regola appresa a una nuova istanza.

In [7]:
def addestra_1R(df, attributi_nominali, attributi_numerici, y_col="y"):
    y = df[y_col]

    def err_di(attr):
        if attr in attributi_numerici:
            med = df[attr].median()
            binned = (df[attr] > med).map({False: "low", True: "high"})
            return regole_e_errori_attributo(binned, y)[1]
        return regole_e_errori_attributo(df[attr], y)[1]

    attrs = pd.Series(attributi_nominali + attributi_numerici)
    errori = attrs.apply(err_di)
    errori.index = attrs
    best = errori.idxmin()

    if best in attributi_numerici:
        med = df[best].median()
        binned = (df[best] > med).map({False: "low", True: "high"})
        regole = pd.crosstab(binned, y).idxmax(axis=1).to_dict()
        return {"attributo": best, "regole": regole, "soglia": med, "errori": int(errori[best])}
    else:
        regole = pd.crosstab(df[best], y).idxmax(axis=1).to_dict()
        return {"attributo": best, "regole": regole, "soglia": None, "errori": int(errori[best])}

In [8]:
def predici_1R(modello, istanza):
    attr = modello["attributo"]
    if modello["soglia"] is not None:                 # attributo numerico
        val = "high" if istanza[attr] > modello["soglia"] else "low"
    else:                                             # attributo nominale
        val = istanza[attr]
    return modello["regole"].get(val, 0)              # default 0 se valore mai visto

In [19]:
# Addestriamo il classificatore sull'intero manuale.csv
modello_1R = addestra_1R(dataFrame_manuale, nominali, numerici)
print("Modello 1R appreso:")
print("  attributo scelto:", modello_1R ["attributo"])
print("  errori sul training:", modello_1R["errori"], "/12")
print("  regole:", modello_1R["regole"])


Modello 1R appreso:
  attributo scelto: job
  errori sul training: 2 /12
  regole: {'admin.': 1, 'blue-collar': 0, 'retired': 1, 'student': 1, 'technician': 1, 'unknown': 0}


## 6. Predizione su tutti i campioni di `manuale.csv`

Come richiesto dalla traccia, valutiamo **sullo stesso file** su cui il modello è stato
costruito. Applichiamo `predici_1R` a ogni riga con `apply(axis=1)` (senza cicli espliciti).

In [10]:
dataFrame_manuale["Predicted"] = dataFrame_manuale.apply(lambda row: predici_1R(modello_1R, row),axis=1)

attributo_scelto = modello_1R["attributo"]
dataFrame_manuale[[attributo_scelto, "y", "Predicted"]]

,job,y,Predicted
0,admin.,1,1
1,blue-collar,0,0
2,blue-collar,0,0
3,retired,1,1
4,admin.,0,1
5,blue-collar,0,0
6,blue-collar,0,0
7,admin.,1,1
8,technician,1,1
9,blue-collar,1,0


## 7. Valutazione delle prestazioni

Usiamo le metriche: **Accuracy, Confusion Matrix,
Precision, Recall, F1-Score**, con classe positiva `y = 1`.

### Accuracy

$$ \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} $$

In [11]:
accuracy = (dataFrame_manuale["y"] == dataFrame_manuale["Predicted"]).mean()
print("Accuracy:", accuracy)

Accuracy: 0.8333333333333334


### Confusion Matrix

Analizza nel dettaglio gli errori, distinguendo falsi positivi e falsi negativi.

In [12]:

tp = len(dataFrame_manuale[(dataFrame_manuale["y"] == 1) & (dataFrame_manuale["Predicted"] == 1)])
tn = len(dataFrame_manuale[(dataFrame_manuale["y"] == 0) & (dataFrame_manuale["Predicted"] == 0)])
fp = len(dataFrame_manuale[(dataFrame_manuale["y"] == 0) & (dataFrame_manuale["Predicted"] == 1)])
fn = len(dataFrame_manuale[(dataFrame_manuale["y"] == 1) & (dataFrame_manuale["Predicted"] == 0)])

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)
confusion_matrix_df

TP = 5
TN = 5
FP = 1
FN = 1


,Predicted 0,Predicted 1
Actual 0,5,1
Actual 1,1,5


### Precision, Recall, F1-Score

$$ \text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad F1 = 2\,\frac{P \cdot R}{P + R} $$

In [13]:
#calcolo dei criteri di valutazione del classificatore
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)


In [ ]:
#Stampa dei risultati finali
print(f"Attributo scelto: {attributo_scelto}")


Attributo scelto: job
Accuracy : 0.8333
Precision: 0.8333
Recall   : 0.8333
F1 Score : 0.8333


## 8. Discussione critica dei risultati

Il classificatore 1R, costruito sull'attributo **`job`** e valutato **sullo stesso
`manuale.csv`**, ottiene:

| Metrica | Valore |
|---|---|
| Accuracy  | 83.33% |
| Precision | 83.33% |
| Recall    | 83.33% |
| F1-Score  | 83.33% |

con matrice di confusione:

|            | Predicted 0 | Predicted 1 |
|------------|:-----------:|:-----------:|
| **Actual 0** | 5 | 1 |
| **Actual 1** | 1 | 5 |

**Lettura dei risultati.**

- 1R classifica correttamente **10 osservazioni su 12** usando **un solo attributo**: per la
  sua semplicità è un risultato notevole, e conferma il ruolo di `job` come predittore.
- Tutte le metriche coincidono (83.33%) perché sui dati bilanciati gli errori si distribuiscono
  simmetricamente (1 FP e 1 FN).

**Il punto critico: overfitting.** Il risultato è **ottimistico** e va interpretato con
cautela. Come osservato nella §4.3, `job` deve parte del suo successo ai **quattro valori con
una sola istanza**, che azzerano gli errori artificialmente. Sullo stesso training questo
gonfia l'accuratezza; su dati nuovi quei valori non offrono alcuna garanzia. È l'**overfitting
da attributi con molti valori**.

**Limiti di questa valutazione.** Valgono le stesse avvertenze del Naïve Bayes: 12 osservazioni,
training = test, metriche puramente **illustrative**. In particolare, valutare 1R sullo stesso
file su cui sceglie l'attributo **non penalizza** l'overfitting — solo una valutazione su dati
separati (o in leave-one-out / holdout) lo farebbe emergere.

